# Setting Up the HF Trainer Object 

In [ ]:
from datasets import load_dataset

dataset = load_dataset("Rogarcia18/symptoms_ner_v00")
# NOTE that the actual labels that will be used for training are under the column: "token_label_ids"
dataset = dataset.rename_column("token_label_ids", "labels")
dataset

/Users/robertagarcia/Desktop/learning/bert_symptom_ner/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['text', 'word_tokens', 'word_labels', 'tokens', 'input_ids', 'token_labels', 'token_label_ids'],
        num_rows: 14288
    })
    validation: Dataset({
        features: ['text', 'word_tokens', 'word_labels', 'tokens', 'input_ids', 'token_labels', 'token_label_ids'],
        num_rows: 1786
    })
    test: Dataset({
        features: ['text', 'word_tokens', 'word_labels', 'tokens', 'input_ids', 'token_labels', 'token_label_ids'],
        num_rows: 1786
    })
})

## Setup Training Arguments

In [ ]:
import os
import json
import numpy as np
from dotenv import load_dotenv
from transformers import (
    TrainingArguments,
    Trainer, 
    AutoTokenizer,
    DistilBertTokenizerFast,
    DistilBertForTokenClassification,
    DataCollatorForTokenClassification,
)

load_dotenv()
os.environ.setdefault("WANDB_PROJECT", "symptom-ner")

# Load id2label and label2id mappings from json files
with open("id2label.json", "r") as f:
    id2label = json.load(f)
with open("label2id.json", "r") as f:
    label2id = json.load(f)

# Set up the tokenizer and  model
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
num_labels = len(id2label)
# USE DistilBertForTokenClassification MODULE FOR NER TASKS
MODEL_NAME = 'distilbert-base-uncased'
model = DistilBertForTokenClassification.from_pretrained(
    pretrained_model_name_or_path=MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
    )

'symptom-ner'

In [ ]:

OUTPUT_DIR = "distilBERT-symptom-ner-v00"
EVAL_STRATEGY = "epoch"  # simple
# Learning rates to experiment with:
LEARNING_RATES = [5e-5, 3e-5, 1e-5]
# Bs to experiment with:
BS = [10, 16, 32, 64]
NUM_EPOCHS = [20]
run_index = 0

# FIRST ARGUMENTS
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy=EVAL_STRATEGY,
    push_to_hub=True,
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,  # 1 epoch likely underfits; 3 is still quick
    weight_decay=0.01,  # same as in video: https://www.youtube.com/watch?v=ujubwa_oa-0
    warmup_ratio=0.1,  # smooth LR start for small data; drop if you want pure simplicity
    save_strategy="epoch",  # align checkpointing with eval
    logging_strategy="epoch",  # keep logging light
    load_best_model_at_end=True,  # restores best checkpoint
    metric_for_best_model="f1",  # set to whatever compute_metrics returns
    report_to=["wandb"],  # enable Weights & Biases logging
    run_name=f"distilbert-symptom-ner-v00-run-{run_index}",  # shows in W&B runs; adjust as needed
)


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Train !

In [ ]:
from metrics import compute_metrics

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics(id2label = id2label),
)

# Uncomment to train
# trainer.train()


/var/folders/9v/0z2cy6c546g6bvh2dzly39x80000gn/T/ipykernel_8407/1412712169.py:13: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
